In [62]:
import pandas as pd
import numpy as np
import re

df = pd.read_csv("/Users/vedheshas/Downloads/enterprise-data-migration /data/legacy_customers.csv")

# Standardize whitespace and empty strings -> NaN
for col in df.columns:
    if df[col].dtype == "object":
        df[col] = df[col].astype(str).str.strip()
        df[col] = df[col].replace({"": np.nan, "None": np.nan, "NULL": np.nan, "nan": np.nan})

df


,customer_id,full_name,email,country,signup_date,credit_score,account_status,kyc_status,last_updated,balance,currency
0,10352,Asha Kumar,ashakumar185@company.com,United States,07/11/2022,200,active,VERIFIED,10-09-2022,1509.02,usd
1,10689,John Rao,johnrao255@company.com,U.S.,2022-05-16,abc,active,VERIFIED,2023-01-13,1786.50,INR
2,10485,Miguel Wilson,miguelwilson940@gmail.com,U.S.,06-06-2022,612,active,VERIFIED,13-10-2022,2243.83,NaN
3,10388,NaN,user875@gmail.com,IN,2022/06/04,679,INACTIVE,VERIFIED,12-10-2022,-173.17,INR
4,10031,Meera Brown,meerabrown570@company.com,USA,24-11-2022,594,active,verified,invalid_date,1902.89,INR
...,...,...,...,...,...,...,...,...,...,...,...
1045,10330,Meera Johnson,meerajohnson556@company.com,IN,2021-13-01,562,ACTIVE,verified,2023/04/06,-12.90,INR
1046,10466,Ravi Iyer,raviiyer584@yahoo.com,United States of America,NaN,750,NaN,PENDING,2022/03/13,4527.48,INR
1047,10121,Li Lee,lilee992@outlook.com,IN,2021/08/19,595,SUSPENDED,VERIFIED,12-08-2024,735.62,EURO
1048,10299,Priya Singh,priyasingh611@gmail.com,United States,2024/01/08,756,NaN,VERIFIED,2023-12-27,3723.63,EURO


In [63]:
# Normalize status + currency
df["account_status"] = df["account_status"].astype(str).str.strip().str.upper().replace({"NAN": np.nan, "NONE": np.nan, "NULL": np.nan})
df["kyc_status"] = df["kyc_status"].astype(str).str.strip().str.upper().replace({"NAN": np.nan, "NONE": np.nan, "NULL": np.nan})
df["currency"] = df["currency"].astype(str).str.strip().str.upper().replace({"NAN": np.nan, "NONE": np.nan, "NULL": np.nan})




In [64]:
# Normalize case first
df["country"] = df["country"].str.strip()

country_map = {
    "US": "USA",
    "U.S.": "USA",
    "United States": "USA",
    "United States of America": "USA",
    "usa": "USA",
    "USA": "USA",
    "india": "India",
    "INDIA": "India",
    "IN": "India",
    "India": "India",
}

df["country"] = df["country"].replace(country_map)

df[["country"]].value_counts(dropna=False)


country
USA        610
India      325
nan        115
Name: count, dtype: int64

In [65]:
def parse_mixed_dates(series: pd.Series) -> pd.Series:
    s = series.astype(str).str.strip()
    s = s.replace({"nan": None, "None": None, "NULL": None})

    # First attempt: let pandas infer automatically
    parsed = pd.to_datetime(s, errors="coerce")

    # Second attempt: day-first formats (for DD-MM-YYYY, DD/MM/YYYY)
    mask = parsed.isna() & s.notna()
    if mask.any():
        parsed.loc[mask] = pd.to_datetime(s[mask], errors="coerce", dayfirst=True)

    return parsed


df["signup_date_parsed"] = parse_mixed_dates(df["signup_date"])
df[["signup_date", "signup_date_parsed"]]
df["last_updated_parsed"] = parse_mixed_dates(df["last_updated"])



/var/folders/r7/8kj70tv15nlbdrzwtjj7p5xw0000gn/T/ipykernel_50015/285763667.py:11: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  parsed.loc[mask] = pd.to_datetime(s[mask], errors="coerce", dayfirst=True)
/var/folders/r7/8kj70tv15nlbdrzwtjj7p5xw0000gn/T/ipykernel_50015/285763667.py:11: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  parsed.loc[mask] = pd.to_datetime(s[mask], errors="coerce", dayfirst=True)


In [66]:
df["balance_num"] = pd.to_numeric(df["balance"], errors="coerce")

allowed_currency = {"USD", "INR"}
df["currency_valid"] = df["currency"].apply(lambda x: True if pd.isna(x) else x in allowed_currency)

def is_valid_balance(bal, original):
    if pd.isna(original):
        return True  # optional
    if pd.isna(bal):
        return False
    return bal >= 0

df["balance_valid"] = [is_valid_balance(b, o) for b, o in zip(df["balance_num"], df["balance"])]


In [67]:
# Email validation (simple but good enough for project)
email_pattern = re.compile(r"^[^@\s]+@[^@\s]+\.[^@\s]+$")

def is_valid_email(x):
    if pd.isna(x):
        return True  # email optional
    return bool(email_pattern.match(str(x)))

df["email_valid"] = df["email"].apply(is_valid_email)

# Credit score validation: optional, but if present must be numeric and in range
df["credit_score_num"] = pd.to_numeric(df["credit_score"], errors="coerce")

def is_valid_credit(score, original):
    if pd.isna(original):
        return True  # missing is allowed
    if pd.isna(score):
        return False # present but not numeric
    return 300 <= score <= 850

df["credit_score_valid"] = [
    is_valid_credit(s, o) for s, o in zip(df["credit_score_num"], df["credit_score"])
]

df[["email", "email_valid", "credit_score", "credit_score_num", "credit_score_valid"]]


,email,email_valid,credit_score,credit_score_num,credit_score_valid
0,ashakumar185@company.com,True,200,200.0,False
1,johnrao255@company.com,True,abc,NaN,False
2,miguelwilson940@gmail.com,True,612,612.0,True
3,user875@gmail.com,True,679,679.0,True
4,meerabrown570@company.com,True,594,594.0,True
...,...,...,...,...,...
1045,meerajohnson556@company.com,True,562,562.0,True
1046,raviiyer584@yahoo.com,True,750,750.0,True
1047,lilee992@outlook.com,True,595,595.0,True
1048,priyasingh611@gmail.com,True,756,756.0,True


In [68]:
def build_reject_reason(row):
    reasons = []

    # customer_id checks
    if pd.isna(row.get("customer_id")):
        reasons.append("missing_customer_id")

    # full_name required
    if pd.isna(row.get("full_name")):
        reasons.append("missing_full_name")

    # date required and must parse
    if pd.isna(row.get("signup_date_parsed")):
        reasons.append("invalid_signup_date")

    # optional but validated if present
    if row.get("email_valid") is False:
        reasons.append("invalid_email")


    # currency check
    if row.get("currency_valid") is False:
        reasons.append("invalid_currency")

    # balance check
    if row.get("balance_valid") is False:
        reasons.append("invalid_balance")

    # cross-field: ACTIVE requires VERIFIED KYC
    acct = row.get("account_status")
    kyc = row.get("kyc_status")
    if acct == "ACTIVE" and kyc != "VERIFIED":
        reasons.append("active_requires_verified_kyc")

    # account_status required
    if pd.isna(row.get("account_status")):
        reasons.append("missing_account_status")

    # kyc_status required
    acct = row.get("account_status")
    kyc = row.get("kyc_status")

# Only enforce KYC if account is ACTIVE
    if acct == "ACTIVE":
        if pd.isna(kyc):
            reasons.append("missing_kyc_for_active_account")
        elif kyc != "VERIFIED":
            reasons.append("active_requires_verified_kyc")

    if row.get("credit_score_valid") is False:
        reasons.append("invalid_credit_score")

    reasons = list(dict.fromkeys(reasons))
    return ";".join(reasons) if reasons else np.nan

df["reject_reason"] = df.apply(build_reject_reason, axis=1)
df[["customer_id","full_name","email","country","signup_date","signup_date_parsed","reject_reason"]]


,customer_id,full_name,email,country,signup_date,signup_date_parsed,reject_reason
0,10352,Asha Kumar,ashakumar185@company.com,USA,07/11/2022,2022-07-11,invalid_credit_score
1,10689,John Rao,johnrao255@company.com,USA,2022-05-16,2022-05-16,invalid_credit_score
2,10485,Miguel Wilson,miguelwilson940@gmail.com,USA,06-06-2022,NaT,invalid_signup_date
3,10388,NaN,user875@gmail.com,India,2022/06/04,NaT,missing_full_name;invalid_signup_date;invalid_...
4,10031,Meera Brown,meerabrown570@company.com,USA,24-11-2022,NaT,invalid_signup_date
...,...,...,...,...,...,...,...
1045,10330,Meera Johnson,meerajohnson556@company.com,India,2021-13-01,NaT,invalid_signup_date;invalid_balance
1046,10466,Ravi Iyer,raviiyer584@yahoo.com,USA,NaN,NaT,invalid_signup_date;missing_account_status
1047,10121,Li Lee,lilee992@outlook.com,India,2021/08/19,NaT,invalid_signup_date;invalid_currency
1048,10299,Priya Singh,priyasingh611@gmail.com,USA,2024/01/08,NaT,invalid_signup_date;invalid_currency;missing_a...


In [69]:
# A simple quality score: more non-null fields + valid parsed date + valid email/credit
df["non_null_count"] = df[["full_name", "email", "country", "signup_date", "credit_score"]].notna().sum(axis=1)
df["date_ok"] = df["signup_date_parsed"].notna().astype(int)
df["email_ok"] = df["email_valid"].astype(int)
df["credit_ok"] = df["credit_score_valid"].astype(int)

df["quality_score"] = df["non_null_count"] + df["date_ok"] + df["email_ok"] + df["credit_ok"]

# Sort so the best record per customer_id comes first
# Sort by customer_id then last_updated_parsed descending (latest first)
df_sorted = df.sort_values(by=["customer_id", "last_updated_parsed"], ascending=[True, False])

dup_mask = df_sorted.duplicated(subset=["customer_id"], keep="first")

df_sorted.loc[dup_mask, "reject_reason"] = df_sorted.loc[dup_mask, "reject_reason"].fillna("")
df_sorted.loc[dup_mask, "reject_reason"] = df_sorted.loc[dup_mask, "reject_reason"].apply(
    lambda x: "duplicate_customer_id" if x == "" else x + ";duplicate_customer_id"
)


df_sorted[["customer_id","quality_score","reject_reason"]]


,customer_id,quality_score,reject_reason
397,10000,4,invalid_signup_date;invalid_balance;missing_ac...
983,10001,8,invalid_currency
162,10002,7,invalid_signup_date
112,10003,7,missing_account_status
899,10004,7,invalid_signup_date
...,...,...,...
490,10995,7,invalid_signup_date
431,10996,6,invalid_signup_date;invalid_currency
483,10997,8,active_requires_verified_kyc
622,10998,7,invalid_signup_date


In [70]:
# Rejected = any row with a reject_reason
rejected = df_sorted[df_sorted["reject_reason"].notna()].copy()
clean = df_sorted[df_sorted["reject_reason"].isna()].copy()

clean_final = clean[[
    "customer_id","full_name","email","country","signup_date_parsed","credit_score_num",
    "account_status","kyc_status","last_updated_parsed","balance_num","currency"
]].copy()

clean_final = clean_final.rename(columns={
    "signup_date_parsed":"signup_date",
    "credit_score_num":"credit_score",
    "last_updated_parsed":"last_updated",
    "balance_num":"balance"
})

clean_final["signup_date"] = pd.to_datetime(clean_final["signup_date"], errors="coerce").dt.strftime("%Y-%m-%d")
clean_final["last_updated"] = pd.to_datetime(clean_final["last_updated"], errors="coerce").dt.strftime("%Y-%m-%d")


# Keep rejected columns + include original and parsed date
rejected_final = rejected[["customer_id", "full_name", "email", "country", "signup_date", "signup_date_parsed", "credit_score", "reject_reason"]].copy()

print("Clean rows:", len(clean_final))
print("Rejected rows:", len(rejected_final))

clean_final.head(), rejected_final.head()


Clean rows: 96
Rejected rows: 954


(     customer_id    full_name                    email country signup_date  \
 145        10012   Alice Shah   aliceshah593@yahoo.com     USA  2023-06-22   
 944        10014  Omar Wilson  omarwilson962@gmail.com     USA  2021-06-25   
 649        10017     Chen Kim     chenkim274@yahoo.com     USA  2023-12-07   
 921        10027     Chen Kim   chenkim259@outlook.com     USA  2022-08-21   
 675        10048     Omar Das     omardas644@yahoo.com     USA  2023-07-09   
 
      credit_score account_status kyc_status last_updated  balance currency  
 145         651.0      SUSPENDED    PENDING          NaN  2177.57      USD  
 944         652.0      SUSPENDED    PENDING          NaN  1105.83      USD  
 649         565.0      SUSPENDED     FAILED   2024-04-20  3232.17      INR  
 921         767.0         ACTIVE   VERIFIED   2024-08-30  1040.43      INR  
 675         600.0      SUSPENDED     FAILED   2024-02-05  2604.92      USD  ,
      customer_id     full_name                       e

In [71]:
clean_final.to_csv("../data/customers_clean.csv", index=False)
rejected_final.to_csv("../data/customers_rejected.csv", index=False)

print("Saved:")
print(" - ../data/customers_clean.csv")
print(" - ../data/customers_rejected.csv")


Saved:
 - ../data/customers_clean.csv
 - ../data/customers_rejected.csv


In [72]:
# Count reasons (explode multi-reasons)
reasons = rejected_final["reject_reason"].str.split(";").explode()
print("Reject reason counts:")
print(reasons.value_counts())


Reject reason counts:
reject_reason
invalid_signup_date               784
active_requires_verified_kyc      247
missing_account_status            207
invalid_currency                  177
missing_kyc_for_active_account     80
invalid_credit_score               57
duplicate_customer_id              50
invalid_email                      48
invalid_balance                    45
missing_full_name                  41
Name: count, dtype: int64


In [73]:
# Sort so latest update comes first
df_sorted = df.sort_values(by=["customer_id", "last_updated_parsed"], ascending=[True, False])

# Mark duplicates beyond the first as rejected
dup_mask = df_sorted.duplicated(subset=["customer_id"], keep="first")

df_sorted.loc[dup_mask, "reject_reason"] = df_sorted.loc[dup_mask, "reject_reason"].fillna("")
df_sorted.loc[dup_mask, "reject_reason"] = df_sorted.loc[dup_mask, "reject_reason"].apply(
    lambda x: "duplicate_customer_id" if x == "" else x + ";duplicate_customer_id"
)

# Now split clean vs rejected
rejected = df_sorted[df_sorted["reject_reason"].notna()].copy()
clean = df_sorted[df_sorted["reject_reason"].isna()].copy()
